# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stg_event_log — 행동 이벤트 스테이징

**목적**: `event_log`(행동 로그 원본)를 분석용 Stage로 정제
**산출물**: `hackle.stg_event_log` (VIEW)
**Grain**: 1행 = 1건의 행동 이벤트
**퍼널 역할**: 리텐션 · 행동 분석 (재방문, 퍼널 등)

**주의**: 이 테이블 자체엔 `user_id`가 없습니다 (`session_id`만 있음). 유저 단위 분석을 하려면
`stg_event_properties`(session↔user 매핑)와 Mart 단계에서 조인해야 합니다. 이 Stage는 조인 없이
`event_log` 원본만 정제합니다.

**핵심 결정 요약(가안 — 아래 확인 셀 돌리고 확정)**
- `event_datetime` UTC → KST 변환, 재방문 dedup용 `event_date_kst`(DATE)도 같이 생성
- `event_key` 종류가 많을 수 있어 분포부터 확인 (자주 쓰는 이벤트가 뭔지 파악)


## Raw 품질 확인

### 1. 컬럼 타입 확인

In [ ]:
%%bigquery df_types --project your-gcp-project

SELECT column_name, data_type
FROM `your-gcp-project.hackle.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'event_log'
ORDER BY ordinal_position

In [ ]:
df_types

In [ ]:
%%bigquery df_id_check --project your-gcp-project

SELECT event_id, id
FROM `your-gcp-project.hackle.event_log`
LIMIT 10

In [ ]:
df_id_check

### 2. PK(event_id) 유니크 · 결측 확인

In [ ]:
%%bigquery df_events_pk_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT event_id) AS distinct_event_id_count,
  COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_event_id_count,
  COUNTIF(event_id IS NULL) AS null_event_id_count,
  COUNTIF(session_id IS NULL) AS null_session_id_count
FROM `your-gcp-project.hackle.event_log`;

In [ ]:
df_events_pk_check

### 3. 전체 컬럼 결측 확인

In [ ]:
%%bigquery df_events_null_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNTIF(event_datetime IS NULL) AS null_event_datetime,
  COUNTIF(event_key IS NULL) AS null_event_key,
  COUNTIF(item_name IS NULL) AS null_item_name,
  COUNTIF(page_name IS NULL) AS null_page_name,
  COUNTIF(friend_count IS NULL) AS null_friend_count,
  COUNTIF(votes_count IS NULL) AS null_votes_count,
  COUNTIF(heart_balance IS NULL) AS null_heart_balance,
  COUNTIF(question_id IS NULL) AS null_question_id
FROM `your-gcp-project.hackle.event_log`;

In [ ]:
df_events_null_check.T.rename(columns={0: 'null_count'})

In [ ]:
%%bigquery df_null_by_event_key --project your-gcp-project

SELECT
  event_key,
  COUNT(*) AS cnt,
  COUNTIF(question_id IS NOT NULL) AS has_question_id,
  COUNTIF(friend_count IS NOT NULL) AS has_friend_count,
  COUNTIF(votes_count IS NOT NULL) AS has_votes_count,
  COUNTIF(heart_balance IS NOT NULL) AS has_heart_balance
FROM `your-gcp-project.hackle.event_log`
GROUP BY event_key
ORDER BY cnt DESC
LIMIT 30

In [ ]:
df_null_by_event_key

In [ ]:
%%bigquery df_question_events --project your-gcp-project

SELECT
  event_key,
  COUNT(*) AS cnt,
  COUNTIF(question_id IS NOT NULL) AS has_question_id
FROM `your-gcp-project.hackle.event_log`
WHERE question_id IS NOT NULL
GROUP BY event_key
ORDER BY cnt DESC

In [ ]:
df_question_events

In [ ]:
%%bigquery df_session_start_check --project your-gcp-project

SELECT
  DATE(event_datetime) AS event_date,
  COUNT(*) AS cnt,
  COUNTIF(friend_count IS NOT NULL) AS has_friend_count
FROM `your-gcp-project.hackle.event_log`
WHERE event_key = '$session_start'
GROUP BY event_date
ORDER BY event_date

In [ ]:
df_session_start_check

In [ ]:
%%bigquery df_session_start_by_login --project your-gcp-project

SELECT
  hp.user_id IS NOT NULL AS has_raw_user_id,
  COUNT(*) AS cnt,
  COUNTIF(he.friend_count IS NOT NULL) AS has_friend_count
FROM `your-gcp-project.hackle.event_log` he
JOIN `your-gcp-project.hackle.event_properties` hp
  ON he.session_id = hp.session_id
WHERE he.event_key = '$session_start'
GROUP BY has_raw_user_id

In [ ]:
df_session_start_by_login

In [ ]:
%%bigquery df_props_dup_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT session_id) AS distinct_session_id_count,
  COUNT(*) - COUNT(DISTINCT session_id) AS duplicate_count
FROM `your-gcp-project.hackle.event_properties`;

In [ ]:
df_props_dup_check

In [ ]:
%%bigquery df_dup_sample --project your-gcp-project

SELECT session_id, COUNT(*) AS cnt
FROM `your-gcp-project.hackle.event_properties`
GROUP BY session_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 5

In [ ]:
df_dup_sample

In [ ]:
%%bigquery df_dup_detail --project your-gcp-project

SELECT *
FROM `your-gcp-project.hackle.event_properties`
WHERE session_id = (
  SELECT session_id
  FROM `your-gcp-project.hackle.event_properties`
  GROUP BY session_id
  HAVING COUNT(*) > 1
  ORDER BY COUNT(*) DESC
  LIMIT 1
)

In [ ]:
df_dup_detail

In [ ]:
%%bigquery df_session_device_check --project your-gcp-project

SELECT
  COUNT(*) AS total_rows,
  COUNTIF(session_id = device_id) AS same_count,
  COUNTIF(session_id != device_id) AS diff_count
FROM `your-gcp-project.hackle.event_properties`

In [ ]:
df_session_device_check

In [ ]:
%%bigquery df_dup_sample --project your-gcp-project

SELECT session_id, COUNT(*) AS cnt
FROM `your-gcp-project.hackle.event_properties`
GROUP BY session_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 5

In [ ]:
df_dup_sample

In [ ]:
%%bigquery df_dup_detail --project your-gcp-project

SELECT *
FROM `your-gcp-project.hackle.event_properties`
WHERE session_id = (
  SELECT session_id
  FROM `your-gcp-project.hackle.event_properties`
  GROUP BY session_id
  HAVING COUNT(*) > 1
  ORDER BY COUNT(*) DESC
  LIMIT 1
)

In [ ]:
df_dup_detail

In [ ]:
%%bigquery df_multi_user_check --project your-gcp-project

SELECT
  COUNT(DISTINCT session_id) AS total_sessions,
  COUNT(DISTINCT IF(user_cnt > 1, session_id, NULL)) AS multi_user_sessions
FROM (
  SELECT session_id, COUNT(DISTINCT user_id) AS user_cnt
  FROM `your-gcp-project.hackle.event_properties`
  GROUP BY session_id
)

In [ ]:
df_multi_user_check

### 4. event_key 분포 (자주 쓰는 이벤트 파악)

In [ ]:
%%bigquery df_event_key --project your-gcp-project

SELECT
  event_key,
  COUNT(*) AS cnt
FROM `your-gcp-project.hackle.event_log`
GROUP BY event_key
ORDER BY cnt DESC
LIMIT 30

In [ ]:
df_event_key

### 5. event_datetime 타임존 분포 확인

In [ ]:
%%bigquery df_event_hour --project your-gcp-project

SELECT
  EXTRACT(HOUR FROM DATETIME(event_datetime, 'Asia/Seoul')) AS hour_kst,
  COUNT(*) AS cnt
FROM `your-gcp-project.hackle.event_log`
GROUP BY hour_kst
ORDER BY hour_kst

In [ ]:
df_event_hour

## 최종 정제 미리보기

In [ ]:
%%bigquery df_events_preview --project your-gcp-project

SELECT
  id,
  event_id,
  session_id,
  event_key,
  item_name,
  page_name,
  friend_count,
  votes_count,
  device_id,
  heart_balance,
  question_id,

  DATETIME(event_datetime, 'Asia/Seoul')       AS event_datetime_kst,
  DATE(DATETIME(event_datetime, 'Asia/Seoul'))  AS event_date_kst   -- 재방문 dedup용
FROM `your-gcp-project.hackle.event_log`
ORDER BY event_datetime DESC
LIMIT 30;

In [ ]:
df_events_preview.head(5)

## 최종 Stage

### view (관례)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_event_log` AS
SELECT
  id,
  event_id,
  session_id,
  event_key,
  item_name,
  page_name,
  friend_count,
  votes_count,
  heart_balance,
  question_id,
  DATETIME(event_datetime, 'Asia/Seoul')       AS event_datetime_kst,
  DATE(DATETIME(event_datetime, 'Asia/Seoul'))  AS event_date_kst
FROM `your-gcp-project.hackle.event_log`;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT event_id) AS distinct_event_id,
  COUNT(DISTINCT event_date_kst) AS distinct_days
FROM `your-gcp-project.stage.stg_event_log`;